In [1]:
import pandas as pd

In [2]:
df = pd.read_pickle(r"C:\code\bachelorarbeit\data\raw\vvz_model.pkl")

In [3]:
df_winf = df[~pd.isna(df["groupId"])]

### Helper functions

```
print_schedule(schedule)
print_offering(offering)
schedule_overlaps(schedule)
```

In [4]:
import datetime
from loguru import logger

def print_schedule(schedule):
    logger.debug(
        f"Schedule:"
    )
    for offering in schedule:
        print_offering(offering)


def print_offering(offering):
    logger.debug(
        f"  Offering {offering.groupId} (LV-ID {offering.courseId}, {offering.ects} ECTS)"
    )
    for date in offering.dates:
        logger.debug(f"    {_format(date['start'])} - {date['end'].strftime('%H:%M')}")


def _format(d: datetime.datetime):
    day = d.strftime("%A")

    return f"{day}{' ' * (10 - len(day))}{d.strftime('%d.%m %H:%M')}"


def schedule_overlaps(schedule) -> bool:
    all_sessions = []
    for offering in schedule:
        all_sessions.extend(offering.dates)

    if not all_sessions:
        return False

    all_sessions.sort(key=lambda x: x["start"])

    for i in range(1, len(all_sessions)):
        prev_session = all_sessions[i - 1]
        curr_session = all_sessions[i]

        if curr_session["start"] < prev_session["end"]:
            logger.debug(f"overlap: {curr_session}, {prev_session}")
            return True

    return False

In [22]:
rounds = 3
objectives = [0.25, 0.33, 0.5, 0.667, 0.75]

import math, json

for o in objectives:
    pct = math.floor(o * 100)
    for i in range(rounds):
        r = i + 1
        obj = {
        "title": f"{pct}% aller Kurse können nicht mehr belegt werden (Zufällige Auswahl).",
            "COURSE_PRIORITY_CONSTRAINTS": { str(cid): -100 for cid in df_winf.sample(frac=o)["courseId"].to_list() },

        "FIXED_TIME_CONSTRAINTS": [
            ["monday", 1, 23, 7],
            ["tuesday", 1, 23, 15],
            ["wednesday", 1, 23, 15],
            ["thursday", 1, 23, 7],
            ["friday", 1, 23, 7],
            ["saturday", 1, 23, -7]
        ],

            }

        with open(fr"C:\Users\Philipp\Documents\WU\bachelorarbeit\models\config\constraint_4442_{pct}_pct_blocked_{r}.json", "w") as f:
            json.dump(obj, f, indent=4)

## Create constraint cases with fixed courses

In [58]:
rounds = 5
objectives = [1, 2, 3, 4, 5, 7, 10]

import math, json, random


def pick_n_courses_from_random_groups(n):
    used = []
    courses = []
    while len(courses) < n:
        groupId = pick_group(used)
        logger.debug(f"scheduled: {len(courses)}")

        for i in range(1000):
            c = pick_course(groupId)
            if not schedule_overlaps([*courses, c]):
                used.append(groupId)
                courses.append(pick_course(groupId))
                break
    return courses

def pick_group(picked_groups):
    groupIds = set(df_winf["groupId"].unique())
    remainingGroupIds = groupIds.difference(set(picked_groups))
    return random.choice(list(remainingGroupIds))

def pick_course(groupId):
    return df_winf[df_winf["groupId"] == groupId].sample(n=1).iloc[0]


for o in objectives:
    for i in range(rounds):
        r = i + 1
        courses = pick_n_courses_from_random_groups(o)
        obj = {
        "title": f"Zahl der fix gesetzten Kurse: {o} (Zufällige Auswahl).",
            "COURSE_PRIORITY_CONSTRAINTS": { str(cid): 100 for cid in courses },

        "FIXED_TIME_CONSTRAINTS": [
            ["monday", 1, 23, 7],
            ["tuesday", 1, 23, 15],
            ["wednesday", 1, 23, 15],
            ["thursday", 1, 23, 7],
            ["friday", 1, 23, 7],
            ["saturday", 1, 23, -7]
        ],

        }

        with open(fr"C:\Users\Philipp\Documents\WU\bachelorarbeit\models\config\constraint_4443_{o}_scheduled_{r}.json", "w") as f:
            json.dump(obj, f, indent=4)

# cs = pick_n_courses_from_random_groups(3)
# print_schedule(cs)
# print(schedule_overlaps(cs))

2026-01-11 11:04:37.560 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 0
2026-01-11 11:04:37.569 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 0
2026-01-11 11:04:37.578 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 0
2026-01-11 11:04:37.591 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 0
2026-01-11 11:04:37.602 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 0
2026-01-11 11:04:37.609 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 0
2026-01-11 11:04:37.611 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 1
2026-01-11 11:04:37.621 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 0
2026-01-11 11:04:37.623 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 1
2026-01-11 11:04:37.633 | DEBUG    | __main__:pick_n_courses_from_random_groups:12 - scheduled: 0
2026-01-11 11:04:37.

KeyboardInterrupt: 

In [14]:
print((f"{df_winf.groupby('groupId').size().value_counts().sort_index()}".split("\n")))

['2     10', '3      4', '5      3', '11     1', '14     1', '22     1', '26     1', '31     1', '33     2', '38     1', '40     1', '42     1', '44     1', 'Name: count, dtype: int64']


381

In [66]:
len(df_winf) / 18

21.166666666666668

In [67]:
len(df_winf["groupId"].unique())

28

In [68]:
381/28

13.607142857142858

In [64]:
len(df_winf) / len(df_winf["groupId"].unique())

13.607142857142858

In [29]:
all_dates = []
for _, row in df_winf.iterrows():
    all_dates.extend(list(map(lambda x: x["start"], row.dates)))

df = pd.to_datetime(all_dates)

# Get the day of the week names
days = df.day_name()

# Calculate percentages
counts = days.value_counts(normalize=True) * 100

# Ensure all days are represented (even if 0%)
all_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
counts = counts.reindex(all_days, fill_value=0)

print(counts)

Monday       21.138452
Tuesday      25.437296
Wednesday    21.849985
Thursday     19.656092
Friday       11.265935
Saturday      0.652238
Sunday        0.000000
Name: proportion, dtype: float64


In [33]:
# 2. Convert to datetime format
df = pd.to_datetime(all_dates)

# 3. Map each timestamp to a 15-minute slot
quarter_hours = df.floor("15min").strftime("%H:%M")

# 4. Calculate percentage per 15-minute slot
quarter_hour_percentages = pd.Series(quarter_hours).value_counts(normalize=True) * 100

# 5. Ensure full range from 07:30 to 22:00 is included
all_slots = pd.date_range("2000-01-01 07:30", "2000-01-01 22:00", freq="15min").strftime("%H:%M")
quarter_hour_percentages = quarter_hour_percentages.reindex(all_slots, fill_value=0)

print(quarter_hour_percentages)

07:30     0.592944
07:45     0.000000
08:00    15.031130
08:15     0.000000
08:30     1.838126
08:45     0.000000
09:00     7.115328
09:15     0.000000
09:30     1.156241
09:45     0.000000
10:00     4.180255
10:15     0.000000
10:30     5.929440
10:45     0.000000
11:00     5.158613
11:15     0.000000
11:30     0.326119
11:45     0.622591
12:00     4.506374
12:15     1.482360
12:30     1.986362
12:45     0.000000
13:00     9.338867
13:15     0.237178
13:30     1.867773
13:45     0.000000
14:00     4.891788
14:15     0.177883
14:30     2.994367
14:45     0.000000
15:00     2.371776
15:15     0.000000
15:30     3.528017
15:45     0.000000
16:00     4.506374
16:15     0.000000
16:30     5.781204
16:45     0.000000
17:00     4.862141
17:15     0.000000
17:30     0.711533
17:45     0.000000
18:00     5.959087
18:15     0.000000
18:30     2.816484
18:45     0.000000
19:00     0.029647
19:15     0.000000
19:30     0.000000
19:45     0.000000
20:00     0.000000
20:15     0.000000
20:30     0.

In [26]:
courses_starting_at_7 = df_winf[
  df_winf["dates"].apply(
    lambda sessions: any(s["start"].hour == 7 for s in sessions if "start" in s)
  )
]

print(courses_starting_at_7[["courseId", "groupId"]])
print(f"Found {len(courses_starting_at_7)} courses with at least one session starting at 07:00.")

     courseId groupId
595      4679     pfo
599      4683     pfo
605      4690     pfo
610      4696     pfo
617      4703     pfo
Found 5 courses with at least one session starting at 07:00.


In [27]:
list(map(lambda x: x["start"], df_winf[df_winf["courseId"] == 4679]["dates"].iloc[0]))

[datetime.datetime(2025, 3, 10, 7, 30),
 datetime.datetime(2025, 3, 17, 7, 30),
 datetime.datetime(2025, 3, 26, 8, 0),
 datetime.datetime(2025, 3, 31, 7, 30),
 datetime.datetime(2025, 4, 2, 8, 0),
 datetime.datetime(2025, 4, 7, 7, 30),
 datetime.datetime(2025, 4, 9, 8, 0),
 datetime.datetime(2025, 4, 23, 8, 0),
 datetime.datetime(2025, 4, 28, 11, 0)]

In [ ]:
# Build (start, end) pairs without overwriting existing `all_dates`
session_intervals = []
for _, row in df_winf.iterrows():
    session_intervals.extend((x["start"], x["end"]) for x in row.dates)

duration_df = pd.DataFrame(session_intervals, columns=["start", "end"])
duration_df["start"] = pd.to_datetime(duration_df["start"])
duration_df["end"] = pd.to_datetime(duration_df["end"])

# Duration in minutes
duration_df["duration_minutes"] = (duration_df["end"] - duration_df["start"]).dt.total_seconds() / 60

# Round to nearest 15 minutes (0.5 rounds up)
duration_df["rounded_15min"] = ((duration_df["duration_minutes"] / 15) + 0.5).astype(int) * 15

# Percentage distribution
distribution_15min = duration_df["rounded_15min"].value_counts(normalize=True).sort_index() * 100
print(distribution_15min)


rounded_15min
60      0.681886
90     11.176994
120    15.209013
135     0.741180
150    33.590276
180     8.301216
210     0.978358
225     0.088942
240    20.189742
255     5.959087
270     1.274830
285     0.059294
300     0.444708
315     0.029647
330     0.444708
345     0.029647
360     0.355766
420     0.118589
450     0.029647
480     0.296472
Name: proportion, dtype: float64


: 

In [5]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

rows = []
for day in day_order:
  starts = []
  ends = []

  for _, row in df_winf.iterrows():
    for session in row["dates"]:
      if session["start"].strftime("%A") == day:
        starts.append(session["start"])
        ends.append(session["end"])

  rows.append({
    "day": day,
    "earliest_start": min(starts).strftime("%H:%M") if starts else None,
    "latest_end": max(ends).strftime("%H:%M") if ends else None,
  })

day_bounds = pd.DataFrame(rows)
print(day_bounds)

         day earliest_start latest_end
0     Monday          12:00      15:00
1    Tuesday          13:00      15:00
2  Wednesday          08:00      17:15
3   Thursday          08:00      20:00
4     Friday          10:00      21:00
5   Saturday          09:00      15:00
6     Sunday            NaN        NaN
